# One-Rec — ranker + item-NCF training (Kaggle)

Trains everything the serving engine needs, using **only signals available at
serving time** (no deprecated Spotify audio-features API):

| Stage | Output | ~Time (T4) |
|---|---|---|
| 1 | `track_meta.parquet` + playlists from the Million Playlist Dataset | 25 min |
| 2 *(optional, off)* | retrained item2vec embeddings | ~4 h CPU |
| 3 | in-notebook Annoy index (feature generation only, not shipped) | 20 min |
| 4 | `ncf_item_v2.pt` — item-NCF (fold-in NeuMF, BPR loss) | 45 min |
| 5 | ranker training set via **serving-faithful retrieval** | 40 min |
| 6 | `lightgbm_ranker_v2.txt` (LambdaRank) + eval vs baselines | 15 min |
| 7 | `artifacts.zip` — download and unpack into the repo's `models/` | 2 min |

**Setup**
1. Attach a Spotify **Million Playlist Dataset** mirror (any dataset containing `mpd.slice.*.json`).
2. Attach your private **`music-rec-base-models`** dataset containing:
   `item2vec.wordvectors`, `item2vec.wordvectors.vectors.npy`, `mood_predictor.pkl`, `features_spec.py`
   (copied from the repo — `features_spec.py` is the exact serving feature contract).
3. Accelerator: **GPU T4**. Run all. Stages checkpoint to `/kaggle/working`, so a
   restarted session resumes instead of recomputing.


In [ ]:
import glob, json, os, pickle, sys, time, zipfile
from pathlib import Path

import numpy as np

# Kaggle ships numpy 2.x — fine here: every artifact we produce (LightGBM text
# model, torch checkpoint, parquet) is format-portable back to the numpy<2
# serving stack. scikit-learn is pinned to match the mood_predictor pickle.
%pip install -q annoy gensim "scikit-learn==1.5.1"
import annoy, gensim, lightgbm as lgb, pandas as pd, sklearn, torch
print("numpy", np.__version__, "| gensim", gensim.__version__, "| sklearn", sklearn.__version__,
      "| lightgbm", lgb.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())


In [ ]:
CFG = dict(
    mpd_glob="/kaggle/input/**/mpd.slice.*.json",
    base_dir=None,                # auto-detected below: dataset containing item2vec.wordvectors
    min_playlist_len=10, max_playlist_len=250, min_artists=3,

    retrain_item2vec=False,       # stage 2 (slow); serving keeps its existing embeddings

    ncf_min_count=15,             # vocab: tracks appearing in >= N playlists
    ncf_gmf_dim=64, ncf_mlp_dim=64, ncf_mlp_layers=[128, 64],
    ncf_epochs=3, ncf_samples_per_epoch=3_000_000, ncf_batch=4096, ncf_lr=1e-3,
    ncf_ctx_min=5, ncf_ctx_max=20,

    ranker_train=25_000, ranker_val=2_500, ranker_test=5_000,
    retrieval_pool=1000, seed_share=0.7, mood_keep=0.55, mood_keep_floor=90, cand_cap=500,
    seed_grade_hi=0.75, seed_grade_lo=0.55,   # graded-label thresholds on seed cosine
    eval_sample_groups=500,

    seed=42,
)

WORK = Path("/kaggle/working")
hits = glob.glob("/kaggle/input/**/item2vec.wordvectors", recursive=True)
assert hits, "attach the music-rec-base-models dataset"
BASE = Path(hits[0]).parent
CFG["base_dir"] = str(BASE)
sys.path.insert(0, str(BASE))
import features_spec as F   # exact copy of services/recommendation/features.py

rng = np.random.default_rng(CFG["seed"])
print("base models:", BASE)
print("feature contract:", F.FEATURE_NAMES)


## Stage 1 — parse MPD → playlists + `track_meta.parquet`

In [ ]:
meta_path, playlists_path = WORK / "track_meta.parquet", WORK / "playlists.npz"

if meta_path.exists() and playlists_path.exists():
    print("stage 1 cached")
else:
    slices = sorted(glob.glob(CFG["mpd_glob"], recursive=True))
    assert slices, "no mpd.slice.*.json found — attach an MPD mirror dataset"
    print(f"{len(slices)} slices")

    meta = {}      # tid -> [name, artist, duration_ms, playlist_count]
    id_of = {}     # tid -> int index
    playlists = []
    t0 = time.time()

    for si, path in enumerate(slices):
        with open(path) as f:
            data = json.load(f)
        for pl in data["playlists"]:
            tracks = pl.get("tracks", [])
            if not (CFG["min_playlist_len"] <= len(tracks) <= CFG["max_playlist_len"]):
                continue
            if len({t["artist_name"] for t in tracks}) < CFG["min_artists"]:
                continue
            ids = np.empty(len(tracks), dtype=np.int32)
            for j, t in enumerate(tracks):
                tid = t["track_uri"].rsplit(":", 1)[-1]
                m = meta.get(tid)
                if m is None:
                    meta[tid] = [t["track_name"], t["artist_name"], t.get("duration_ms", 0) or 0, 1]
                    id_of[tid] = len(id_of)
                else:
                    m[3] += 1
                ids[j] = id_of[tid]
            playlists.append(ids)
        if si % 100 == 0:
            print(f"  {si}/{len(slices)} slices | {len(playlists):,} playlists | "
                  f"{len(meta):,} tracks | {time.time()-t0:.0f}s", flush=True)

    tids = list(id_of.keys())  # insertion order == index order
    pd.DataFrame({
        "track_id": tids,
        "name": [meta[t][0] for t in tids],
        "artist": [meta[t][1] for t in tids],
        "duration_ms": np.array([meta[t][2] for t in tids], dtype=np.int64),
        "playlist_count": np.array([meta[t][3] for t in tids], dtype=np.int32),
    }).to_parquet(meta_path, index=False)

    flat = np.concatenate(playlists)
    lens = np.array([len(p) for p in playlists], dtype=np.int32)
    np.savez(playlists_path, flat=flat, lens=lens)
    del meta, id_of, playlists
    print(f"saved {len(lens):,} playlists, {len(tids):,} tracks")

meta_df = pd.read_parquet(meta_path)
z = np.load(playlists_path)
flat, lens = z["flat"], z["lens"]
offsets = np.zeros(len(lens) + 1, dtype=np.int64)
np.cumsum(lens, out=offsets[1:])
def playlist_at(i):
    return flat[offsets[i]:offsets[i + 1]]
print(f"{len(lens):,} playlists | {len(meta_df):,} tracks")


## Stage 2 *(optional)* — retrain item2vec

Off by default: serving keeps its existing 597K-track embeddings + ANN index.

In [ ]:
if CFG["retrain_item2vec"]:
    from gensim.models import Word2Vec
    tid_arr_tmp = meta_df["track_id"].to_numpy()
    sentences = ([tid_arr_tmp[playlist_at(i)].tolist() for i in range(len(lens))])
    w2v = Word2Vec(sentences, vector_size=200, window=50, min_count=5, sg=1,
                   negative=10, workers=4, epochs=5, seed=CFG["seed"])
    w2v.wv.save(str(WORK / "item2vec_v2.wordvectors"))
    print("retrained item2vec:", len(w2v.wv))
else:
    print("skipped (CFG['retrain_item2vec']=False)")


## Stage 3 — load embeddings, build the in-notebook Annoy index

In [ ]:
from gensim.models import KeyedVectors
from annoy import AnnoyIndex

wv = KeyedVectors.load(str(BASE / "item2vec.wordvectors"), mmap="r")
DIM = wv.vector_size
vecs = np.asarray(wv.vectors, dtype=np.float32)
norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms == 0] = 1.0
unit_vecs = vecs / norms
print(f"{len(wv):,} embeddings, dim={DIM}")

# MPD index <-> item2vec row alignment
tid_arr = meta_df["track_id"].to_numpy()
i2v_row = np.array([wv.key_to_index.get(t, -1) for t in tid_arr], dtype=np.int64)
mpd_of_row = np.full(len(wv), -1, dtype=np.int64)
mapped = np.where(i2v_row >= 0)[0]
mpd_of_row[i2v_row[mapped]] = mapped
print(f"{len(mapped):,}/{len(meta_df):,} MPD tracks have embeddings; "
      f"{(mpd_of_row >= 0).sum():,}/{len(wv):,} embeddings have MPD metadata")

artist_norm = meta_df["artist"].fillna("").str.lower().str.strip().to_numpy()
pop = meta_df["playlist_count"].to_numpy()
log_pop_all = (np.log1p(pop) / np.log1p(pop.max())).astype(np.float32)
dur_all = meta_df["duration_ms"].fillna(0).to_numpy(np.float32)

ann_path = WORK / "ann.index"
ann = AnnoyIndex(DIM, "angular")
if ann_path.exists():
    ann.load(str(ann_path)); print("annoy cached")
else:
    t0 = time.time()
    for i in range(len(wv)):
        ann.add_item(i, vecs[i])
    ann.build(32)
    ann.save(str(ann_path))
    print(f"annoy built in {time.time()-t0:.0f}s")

mp = pickle.load(open(BASE / "mood_predictor.pkl", "rb"))
mood_model = mp["model"]
mood_order = [mp.get("feature_names", F.MOOD_DIMS).index(d) for d in F.MOOD_DIMS]
def predict_mood(rows):
    preds = np.clip(mood_model.predict(vecs[rows]), 0.0, 1.0)
    return preds[:, mood_order].astype(np.float32)


## Stage 4 — item-NCF (fold-in NeuMF, BPR)

Item embedding tables only — no playlist tower — so serving can score unseen
playlists by mean-pooling their tracks ("fold-in"). Architecture and checkpoint
format match `services/recommendation/ncf.py` exactly (`item-ncf-v2`).


In [ ]:
ncf_path = WORK / "ncf_item_v2.pt"

# Kaggle sometimes assigns a P100 (sm_60), which current torch wheels no longer
# support — smoke-test CUDA and fall back to CPU (the NCF is small enough).
device = "cpu"
if torch.cuda.is_available():
    try:
        (torch.ones(2, device="cuda") * 2).sum().item()
        device = "cuda"
    except Exception as exc:
        print(f"CUDA present but unusable ({type(exc).__name__}) — training on CPU")
print("NCF device:", device)

vocab_mpd = np.where(pop >= CFG["ncf_min_count"])[0]
ncf_index_of = np.full(len(meta_df), -1, dtype=np.int64)
ncf_index_of[vocab_mpd] = np.arange(len(vocab_mpd))
N_NCF = len(vocab_mpd)
print(f"NCF vocab: {N_NCF:,} tracks (>= {CFG['ncf_min_count']} playlist occurrences)")

NCF_CONFIG = {"n_tracks": N_NCF, "gmf_dim": CFG["ncf_gmf_dim"],
              "mlp_dim": CFG["ncf_mlp_dim"], "mlp_layers": CFG["ncf_mlp_layers"]}

# Must mirror services/recommendation/ncf.py::_build_model (state_dict compatible).
class ItemNCF(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        nn = torch.nn
        n = config["n_tracks"]
        self.gmf_emb = nn.Embedding(n, config["gmf_dim"])
        self.mlp_emb = nn.Embedding(n, config["mlp_dim"])
        layers, in_dim = [], config["mlp_dim"] * 2
        for out_dim in config["mlp_layers"]:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU()]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.head = nn.Linear(config["gmf_dim"] + in_dim, 1)

def score_batch(model, ctx_pad, ctx_mask, cand):
    m = ctx_mask.unsqueeze(-1)
    denom = ctx_mask.sum(1, keepdim=True).clamp(min=1.0)
    u_gmf = (model.gmf_emb(ctx_pad) * m).sum(1) / denom
    u_mlp = (model.mlp_emb(ctx_pad) * m).sum(1) / denom
    gmf_out = u_gmf * model.gmf_emb(cand)
    mlp_out = model.mlp(torch.cat([u_mlp, model.mlp_emb(cand)], dim=1))
    return model.head(torch.cat([gmf_out, mlp_out], dim=1)).squeeze(1)


In [ ]:
if ncf_path.exists():
    print("stage 4 cached")
else:
    # Playlists usable for NCF: >= ctx_min+1 vocab tracks
    pl_vocab = []
    for i in range(len(lens)):
        v = ncf_index_of[playlist_at(i)]
        v = v[v >= 0]
        if len(v) >= CFG["ncf_ctx_min"] + 1:
            pl_vocab.append(v.astype(np.int64))
    print(f"{len(pl_vocab):,} playlists usable for NCF training")

    popular_pool = ncf_index_of[vocab_mpd[np.argsort(-pop[vocab_mpd])[:max(N_NCF // 5, 1)]]]

    def gen_batch(B):
        L = CFG["ncf_ctx_max"]
        ctx_pad = np.zeros((B, L), dtype=np.int64)
        ctx_mask = np.zeros((B, L), dtype=np.float32)
        pos = np.empty(B, dtype=np.int64)
        neg = np.empty(B, dtype=np.int64)
        pls = rng.integers(0, len(pl_vocab), B)
        half = rng.random(B) < 0.5
        neg[half] = rng.choice(popular_pool, half.sum())
        neg[~half] = rng.integers(0, N_NCF, (~half).sum())
        for b, pi in enumerate(pls):
            tracks = pl_vocab[pi]
            k = int(rng.integers(CFG["ncf_ctx_min"], min(CFG["ncf_ctx_max"], len(tracks) - 1) + 1))
            picks = rng.choice(len(tracks), k + 1, replace=False)
            pos[b] = tracks[picks[0]]
            ctx = tracks[picks[1:]]
            ctx_pad[b, :len(ctx)] = ctx
            ctx_mask[b, :len(ctx)] = 1.0
        return (torch.from_numpy(ctx_pad).to(device), torch.from_numpy(ctx_mask).to(device),
                torch.from_numpy(pos).to(device), torch.from_numpy(neg).to(device))

    model = ItemNCF(NCF_CONFIG).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["ncf_lr"])
    steps_per_epoch = CFG["ncf_samples_per_epoch"] // CFG["ncf_batch"]

    for epoch in range(CFG["ncf_epochs"]):
        model.train(); running = 0.0; t0 = time.time()
        for step in range(steps_per_epoch):
            ctx_pad, ctx_mask, pos_t, neg_t = gen_batch(CFG["ncf_batch"])
            s_pos = score_batch(model, ctx_pad, ctx_mask, pos_t)
            s_neg = score_batch(model, ctx_pad, ctx_mask, neg_t)
            loss = -torch.nn.functional.logsigmoid(s_pos - s_neg).mean()  # BPR
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item()
            if (step + 1) % 200 == 0:
                print(f"  epoch {epoch+1} step {step+1}/{steps_per_epoch} "
                      f"loss {running/200:.4f} ({time.time()-t0:.0f}s)", flush=True)
                running = 0.0

    model.eval().cpu()
    torch.save({
        "format": "item-ncf-v2",
        "state_dict": model.state_dict(),
        "track_id_map": {tid_arr[mpd]: int(i) for i, mpd in enumerate(vocab_mpd)},
        "config": NCF_CONFIG,
    }, ncf_path)
    print("saved", ncf_path)


In [ ]:
# Fold-in scorer used during ranker-dataset generation (mirrors serving semantics)
ckpt = torch.load(ncf_path, map_location="cpu", weights_only=False)
assert ckpt["format"] == "item-ncf-v2"
ncf_model = ItemNCF(ckpt["config"])
ncf_model.load_state_dict(ckpt["state_dict"])
ncf_model.eval()

@torch.inference_mode()
def ncf_score(context_mpd, cand_mpd):
    ctx = ncf_index_of[context_mpd]; ctx = ctx[ctx >= 0]
    if len(ctx) < 3:
        return None, None
    cand = ncf_index_of[cand_mpd]
    ok = cand >= 0
    scores = np.zeros(len(cand_mpd), dtype=np.float32)
    if ok.any():
        L = len(ctx)
        ctx_pad = torch.from_numpy(ctx).unsqueeze(0)
        ctx_mask = torch.ones(1, L)
        cand_t = torch.from_numpy(cand[ok])
        m = ctx_mask.unsqueeze(-1)
        u_gmf = (ncf_model.gmf_emb(ctx_pad) * m).sum(1) / L
        u_mlp = (ncf_model.mlp_emb(ctx_pad) * m).sum(1) / L
        gmf_out = u_gmf * ncf_model.gmf_emb(cand_t)
        mlp_out = ncf_model.mlp(torch.cat([u_mlp.expand(len(cand_t), -1), ncf_model.mlp_emb(cand_t)], dim=1))
        s = ncf_model.head(torch.cat([gmf_out, mlp_out], dim=1)).squeeze(1)
        scores[ok] = torch.sigmoid(s).numpy()
    return scores, ok.astype(np.float32)


## Stage 5 — ranker training set (serving-faithful leave-N-out)

Per playlist: context = 80% of its in-vocab tracks, **seed = one held-out
track, positives = the rest of the holdout**. Candidates come from the *exact
serving retrieval* — 70% seed ANN / 30% playlist-mean ANN, artist pre-dedup,
mood filter — and features from `features_spec.build_matrix`, so the model
trains on precisely the distribution it will score at serving time.

**Seed-graded relevance:** positives are graded 1/2/3 by their embedding
cosine to the seed, so LambdaRank learns to continue the playlist *in the
seed's sonic direction* instead of treating all continuations as equal.


In [ ]:
rng5 = np.random.default_rng(CFG["seed"] + 5)

eligible = [i for i in range(len(lens)) if (i2v_row[playlist_at(i)] >= 0).sum() >= 12]
print(f"{len(eligible):,} eligible playlists")
perm = rng5.permutation(len(eligible))
need = CFG["ranker_train"] + CFG["ranker_val"] + CFG["ranker_test"]
assert len(eligible) >= need, "not enough playlists — lower the sample sizes in CFG"
splits = {
    "train": [eligible[j] for j in perm[:CFG["ranker_train"]]],
    "val":   [eligible[j] for j in perm[CFG["ranker_train"]:CFG["ranker_train"] + CFG["ranker_val"]]],
    "test":  [eligible[j] for j in perm[CFG["ranker_train"] + CFG["ranker_val"]:need]],
}

K_SEED = int(CFG["retrieval_pool"] * CFG["seed_share"])
K_PLAYLIST = CFG["retrieval_pool"] - K_SEED


def build_group(pl_idx):
    tracks_mpd = playlist_at(pl_idx)
    tracks_mpd = tracks_mpd[i2v_row[tracks_mpd] >= 0]
    order = rng5.permutation(len(tracks_mpd))
    n_hold = max(2, int(0.2 * len(order)))
    holdout, context = tracks_mpd[order[:n_hold]], tracks_mpd[order[n_hold:]]
    seed_mpd = int(holdout[0])
    positives = set(holdout[1:].tolist())

    # --- retrieval (mirrors engine._retrieve) ---
    seed_row = int(i2v_row[seed_mpd])
    seed_nb = [r for r in ann.get_nns_by_item(seed_row, K_SEED + 1) if r != seed_row][:K_SEED]
    ctx_rows = i2v_row[context]
    mean_vec = vecs[ctx_rows].mean(axis=0)
    pl_nb = ann.get_nns_by_vector(mean_vec, K_PLAYLIST)

    seed_rank = {r: k for k, r in enumerate(seed_nb)}
    pl_rank = {r: k for k, r in enumerate(pl_nb)}
    exclude = set(ctx_rows.tolist()); exclude.add(seed_row)
    cand_rows, seen = [], set()
    for r in [*seed_nb, *pl_nb]:
        if r not in exclude and r not in seen:
            seen.add(r); cand_rows.append(r)
    if not cand_rows:
        return None
    cand_rows = np.array(cand_rows, dtype=np.int64)
    cand_mpd = mpd_of_row[cand_rows]

    # --- artist pre-dedup (keep best-retrieved per known artist) ---
    cand_artists = np.where(cand_mpd >= 0, artist_norm[np.maximum(cand_mpd, 0)], "")
    keep, seen_a = [], set()
    for j, a in enumerate(cand_artists):
        if a:
            if a in seen_a:
                continue
            seen_a.add(a)
        keep.append(j)
    cand_rows, cand_mpd, cand_artists = cand_rows[keep], cand_mpd[keep], cand_artists[keep]

    # --- mood filter (mirrors engine._mood_filter with limit=30) ---
    moods = predict_mood(cand_rows)
    seed_mood = predict_mood(np.array([seed_row]))[0]
    ctx_mood = predict_mood(ctx_rows[:50]).mean(axis=0)
    target_mood = 0.8 * seed_mood + 0.2 * ctx_mood
    mood_sim = 1.0 - np.abs(moods - target_mood).mean(axis=1)
    keep_n = max(min(len(cand_rows), CFG["mood_keep_floor"]), int(len(cand_rows) * CFG["mood_keep"]))
    keep = np.sort(np.argsort(-mood_sim)[:keep_n])[:CFG["cand_cap"]]
    cand_rows, cand_mpd, cand_artists, moods = cand_rows[keep], cand_mpd[keep], cand_artists[keep], moods[keep]

    is_pos = np.array([m in positives for m in cand_mpd])
    if not is_pos.any():
        return None   # no retrieved positives — reported as retrieval misses

    # Graded relevance: positives closer to the seed's sound are worth more
    # (label 3 > 2 > 1), teaching LambdaRank to continue the playlist in the
    # seed's direction rather than treating all continuations as equal.
    seed_sim = unit_vecs[cand_rows] @ unit_vecs[seed_row]
    labels = np.where(is_pos & (seed_sim >= CFG["seed_grade_hi"]), 3.0,
             np.where(is_pos & (seed_sim >= CFG["seed_grade_lo"]), 2.0,
             np.where(is_pos, 1.0, 0.0))).astype(np.float32)

    # --- features (identical to serving) ---
    known = cand_mpd >= 0
    ncf_s, ncf_m = ncf_score(np.concatenate([context, [seed_mpd]]), np.maximum(cand_mpd, 0))
    if ncf_s is not None:
        ncf_s, ncf_m = np.where(known, ncf_s, 0.0), np.where(known, ncf_m, 0.0)

    ctx_shares = pd.Series(artist_norm[context]).value_counts() / len(context)
    dur_ctx = dur_all[context]; dur_ctx = dur_ctx[dur_ctx > 0]
    ctx_obj = F.RankingContext(
        seed_vec=vecs[seed_row],
        playlist_mean_vec=mean_vec,
        playlist_vecs=vecs[ctx_rows[:100]],
        seed_rank={int(r): k for r, k in seed_rank.items()},
        playlist_rank={int(r): k for r, k in pl_rank.items()},
        seed_artist=str(artist_norm[seed_mpd]),
        playlist_artist_share={a: float(s) for a, s in ctx_shares.items() if a},
        target_mood=target_mood,
        playlist_mean_duration_ms=float(dur_ctx.mean()) if len(dur_ctx) else 0.0,
    )
    X = F.build_matrix(
        ids=[int(r) for r in cand_rows],
        vectors=vecs[cand_rows],
        artists=list(cand_artists),
        log_pop=np.where(known, log_pop_all[np.maximum(cand_mpd, 0)], 0.0).astype(np.float32),
        durations_ms=np.where(known, dur_all[np.maximum(cand_mpd, 0)], 0.0).astype(np.float32),
        moods=moods,
        ncf_scores=ncf_s, ncf_mask=ncf_m,
        ctx=ctx_obj,
    )
    return X, labels, len(holdout) - 1


def build_split(name, playlist_ids):
    cache = WORK / f"ranker_{name}.npz"
    if cache.exists():
        z = np.load(cache)
        return z["X"], z["y"], z["g"]
    Xs, ys, gs, misses, t0 = [], [], [], 0, time.time()
    for n, pi in enumerate(playlist_ids):
        out = build_group(pi)
        if out is None:
            misses += 1
            continue
        X, y, _ = out
        Xs.append(X.to_numpy(np.float32)); ys.append(y); gs.append(len(y))
        if (n + 1) % 2000 == 0:
            print(f"  {name}: {n+1}/{len(playlist_ids)} groups "
                  f"({time.time()-t0:.0f}s, {misses} retrieval misses)", flush=True)
    X, y, g = np.concatenate(Xs), np.concatenate(ys), np.array(gs, dtype=np.int32)
    np.savez(cache, X=X, y=y, g=g)
    print(f"{name}: {len(g):,} groups, {len(y):,} rows, retrieval recall "
          f"{(1 - misses/len(playlist_ids)):.1%} of groups had >=1 retrieved positive")
    return X, y, g

X_train, y_train, g_train = build_split("train", splits["train"])
X_val, y_val, g_val = build_split("val", splits["val"])
X_test, y_test, g_test = build_split("test", splits["test"])


## Stage 6 — LightGBM LambdaRank + evaluation vs baselines

In [ ]:
params = dict(objective="lambdarank", metric="ndcg", ndcg_eval_at=[10, 50],
              learning_rate=0.05, num_leaves=63, min_data_in_leaf=50,
              feature_fraction=0.9, lambdarank_truncation_level=50,
              verbosity=-1, seed=CFG["seed"])

train_ds = lgb.Dataset(X_train, label=y_train.astype(np.int32), group=g_train, feature_name=F.FEATURE_NAMES)
val_ds = lgb.Dataset(X_val, label=y_val.astype(np.int32), group=g_val, feature_name=F.FEATURE_NAMES, reference=train_ds)

booster = lgb.train(params, train_ds, num_boost_round=500, valid_sets=[val_ds],
                    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(25)])
booster.save_model(str(WORK / "lightgbm_ranker_v2.txt"))
assert lgb.Booster(model_file=str(WORK / "lightgbm_ranker_v2.txt")).feature_name() == F.FEATURE_NAMES
print("trees:", booster.num_trees())

importance = dict(zip(F.FEATURE_NAMES, booster.feature_importance("gain").round(1).tolist()))
print(json.dumps(dict(sorted(importance.items(), key=lambda kv: -kv[1])), indent=2))


In [ ]:
def group_metrics(scores, labels, seed_cos, ks=(10, 50)):
    order = np.argsort(-scores)
    rel = labels[order]
    out = {}
    for k in ks:
        topk = rel[:k]
        out[f"recall@{k}"] = float(topk.sum() / labels.sum())
        dcg = float((topk / np.log2(np.arange(2, len(topk) + 2))).sum())
        ideal = np.sort(labels)[::-1][:k]
        idcg = float((ideal / np.log2(np.arange(2, len(ideal) + 2))).sum())
        out[f"ndcg@{k}"] = dcg / idcg if idcg > 0 else 0.0
    out["seed_cos@10"] = float(seed_cos[order][:10].mean())
    return out

seed_cos_col = F.FEATURE_NAMES.index("seed_i2v_cos")
log_pop_col = F.FEATURE_NAMES.index("log_pop")
scorers = {
    "lightgbm_v2": lambda X: booster.predict(X),
    "seed_cosine_only": lambda X: X[:, seed_cos_col],
    "popularity_only": lambda X: X[:, log_pop_col],
    "retrieval_order": lambda X: -np.arange(len(X), dtype=np.float32),
}

# Metrics use binarized labels so results stay comparable with the
# previous (binary-label) training run; seed_cos@10 shows the vibe shift.
results = {name: [] for name in scorers}
start = 0
for gl in g_test:
    Xg = X_test[start:start + gl]
    yb = (y_test[start:start + gl] > 0).astype(np.float64)
    cosg = Xg[:, seed_cos_col]
    for name, fn in scorers.items():
        results[name].append(group_metrics(np.asarray(fn(Xg), dtype=np.float64), yb, cosg))
    start += gl

metrics = {name: {k: float(np.mean([m[k] for m in ms])) for k in ms[0]}
           for name, ms in results.items()}
print(pd.DataFrame(metrics).T.round(4))

json.dump({"metrics": metrics, "feature_importance_gain": importance,
           "config": {k: v for k, v in CFG.items() if isinstance(v, (int, float, str, bool))},
           "dataset": {"train_groups": int(len(g_train)), "val_groups": int(len(g_val)),
                        "test_groups": int(len(g_test)), "ncf_vocab": int(N_NCF)}},
          open(WORK / "metrics.json", "w"), indent=2)


## Stage 7 — package `artifacts.zip`

Download it, then in the repo run `python scripts/install_artifacts.py artifacts.zip`.

In [ ]:
# eval_sample.parquet: feature rows + labels for the first N test groups (local A/B)
n_rows = int(g_test[:CFG["eval_sample_groups"]].sum())
sample = pd.DataFrame(X_test[:n_rows], columns=F.FEATURE_NAMES)
sample["label"] = y_test[:n_rows]
sample["qid"] = np.repeat(np.arange(CFG["eval_sample_groups"]), g_test[:CFG["eval_sample_groups"]])
sample.to_parquet(WORK / "eval_sample.parquet", index=False)

json.dump(F.FEATURE_NAMES, open(WORK / "feature_names.json", "w"))

with zipfile.ZipFile(WORK / "artifacts.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["lightgbm_ranker_v2.txt", "ncf_item_v2.pt", "track_meta.parquet",
              "metrics.json", "feature_names.json", "eval_sample.parquet"]:
        zf.write(WORK / f, f)
print("artifacts.zip:", f"{(WORK / 'artifacts.zip').stat().st_size / 1e6:.1f} MB")
